# cGAN model with Adversarial Loss for generation of Synthtic Datasets

In [2]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import TensorBoard
import datetime

gpus = tf.config.list_physical_devices('GPU')
if gpus:
  print(f"GPU is available and will be used: {gpus[0]}")
else:
  print("No GPU found. TensorFlow will use the CPU.")


# --- 1. Load and Preprocess Data ---

# Load the CIFAR-10 dataset
(x_train, y_train), (_, _) = tf.keras.datasets.cifar10.load_data()

# Normalize pixel values to the [-1, 1] range.
# This is crucial because the generator's final 'tanh' activation
# outputs values in this exact range.
x_train = (x_train.astype('float32') - 127.5) / 127.5

print("Shape of training images:", x_train.shape)
print("Shape of training labels:", y_train.shape)


# --- 2. Define Model Constants ---
IMG_SHAPE = (32, 32, 3)
NUM_CLASSES = 10
LATENT_DIM = 100


No GPU found. TensorFlow will use the CPU.


2025-07-26 22:19:08.879579: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Shape of training images: (50000, 32, 32, 3)
Shape of training labels: (50000, 1)


#### Building the Generator

In [3]:
from tensorflow.keras.layers import Input, Dense, Reshape, Flatten, Concatenate, Embedding
from tensorflow.keras.layers import Conv2D, Conv2DTranspose, LeakyReLU
from tensorflow.keras.models import Model

def build_generator():
    """Builds the Generator model that creates images from noise and labels."""
    # Input for the random noise vector
    noise_input = Input(shape=(LATENT_DIM,))

    # Input for the class label (a single integer)
    label_input = Input(shape=(1,))
    
    # Embedding layer turns the label integer into a dense vector
    label_embedding = Embedding(NUM_CLASSES, 50)(label_input)
    # Reshape the label embedding to be combined with the noise
    label_embedding = Dense(8 * 8)(label_embedding)
    label_embedding = Reshape((8, 8, 1))(label_embedding)

    # Process the noise into a small feature map
    noise = Dense(128 * 8 * 8, activation='relu')(noise_input)
    noise = Reshape((8, 8, 128))(noise)

    # Combine the processed noise and label maps
    concatenated_input = Concatenate()([noise, label_embedding])

    # Use Conv2DTranspose layers to upsample the feature map into a full-sized image
    x = Conv2DTranspose(128, kernel_size=4, strides=2, padding='same', activation='relu')(concatenated_input)
    x = Conv2DTranspose(128, kernel_size=4, strides=2, padding='same', activation='relu')(x)
    
    # The final output layer uses a 'tanh' activation to produce an image
    # with pixel values between -1 and 1, matching our preprocessed data.
    x = Conv2D(3, kernel_size=5, padding='same', activation='tanh')(x)

    # Create and return the final model
    generator = Model([noise_input, label_input], x, name="generator")
    return generator


#### Building the Discriminator

In [4]:
def build_discriminator():
    """Builds the Discriminator model that classifies images as real or fake."""
    # Input for the image
    img_input = Input(shape=IMG_SHAPE)

    # Input for the class label
    label_input = Input(shape=(1,))
    
    # Process the label into a feature map that can be combined with the image
    label_embedding = Embedding(NUM_CLASSES, 50)(label_input)
    label_embedding = Dense(IMG_SHAPE[0] * IMG_SHAPE[1])(label_embedding)
    label_embedding = Reshape((IMG_SHAPE[0], IMG_SHAPE[1], 1))(label_embedding)

    # Combine the label embedding and the image as input channels
    concatenated_input = Concatenate()([img_input, label_embedding])

    # CNN layers to extract features from the input
    x = Conv2D(64, kernel_size=3, strides=2, padding='same')(concatenated_input)
    x = LeakyReLU(alpha=0.2)(x)
    x = Conv2D(128, kernel_size=3, strides=2, padding='same')(x)
    x = LeakyReLU(alpha=0.2)(x)
    x = Flatten()(x)
    
    # The final output layer uses a 'sigmoid' activation to produce a
    # probability score between 0 (fake) and 1 (real).
    x = Dense(1, activation='sigmoid')(x)

    # Create and return the final model
    discriminator = Model([img_input, label_input], x, name="discriminator")
    return discriminator

#### Compiling the Combined cGAN Model

In [5]:
from tensorflow.keras.optimizers import Adam

# Build and compile the discriminator
discriminator = build_discriminator()
discriminator.compile(loss='binary_crossentropy', optimizer=Adam(0.0002, 0.5), metrics=['accuracy'])

# Build the generator
generator = build_generator()

# For the combined model, we only train the generator.
# We "freeze" the discriminator's weights.
discriminator.trainable = False

# Define the inputs for the combined model
noise_input = Input(shape=(LATENT_DIM,))
label_input = Input(shape=(1,))

# The generator produces an image from the inputs
generated_img = generator([noise_input, label_input])

# The discriminator evaluates this generated image
validity = discriminator([generated_img, label_input])

# The combined model chains the generator and discriminator.
# Its purpose is to train the generator to fool the discriminator.
cgan = Model([noise_input, label_input], validity, name="cgan")
cgan.compile(loss='binary_crossentropy', optimizer=Adam(0.0002, 0.5))

print("--- Generator Summary ---")
generator.summary()
print("\n--- Discriminator Summary ---")
discriminator.summary()
print("\n--- Combined GAN Summary ---")
cgan.summary()

--- Generator Summary ---


/home/taz/Documents/Masters/DeepLearning/.venv/lib/python3.12/site-packages/keras/src/layers/activations/leaky_relu.py:41: UserWarning: Argument `alpha` is deprecated. Use `negative_slope` instead.
  warnings.warn(


Model: "generator"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_3       │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_2       │ (None, 100)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_1         │ (None, 1, 50)     │        500 │ input_layer_3[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 8192)      │    827,392 │ input_layer_2[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 1, 64)     │      3,264 │ embedding_1[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape_2 (Reshape) │ (None, 8, 8, 128) │          0 │ dense_3[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape_1 (Reshape) │ (None, 8, 8, 1)   │          0 │ dense_2[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_1       │ (None, 8, 8, 129) │          0 │ reshape_2[0][0],  │
│ (Concatenate)       │                   │            │ reshape_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_transpose    │ (None, 16, 16,    │    264,320 │ concatenate_1[0]… │
│ (Conv2DTranspose)   │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_transpose_1  │ (None, 32, 32,    │    262,272 │ conv2d_transpose… │
│ (Conv2DTranspose)   │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 32, 32, 3) │      9,603 │ conv2d_transpose… │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 1,367,351 (5.22 MB)

...

Model: "cgan"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_4       │ (None, 100)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_5       │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ generator           │ (None, 32, 32, 3) │  1,367,351 │ input_layer_4[0]… │
│ (Functional)        │                   │            │ input_layer_5[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ discriminator       │ (None, 1)         │    137,141 │ generator[0][0],  │
│ (Functional)        │                   │            │ input_layer_5[0]… │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 1,504,492 (5.74 MB)

 Trainable params: 1,367,351 (5.22 MB)

 Non-trainable params: 137,141 (535.71 KB)

#### The Training Loop

In [ ]:
def sample_images(epoch, generator, latent_dim):
    """A helper function to generate and display a grid of images for visual inspection."""
    r, c = 2, 5
    noise = np.random.normal(0, 1, (r * c, latent_dim))
    # Generate one image for each of the 10 classes
    sampled_labels = np.arange(0, 10).reshape(-1, 1)
    
    gen_imgs = generator.predict([noise, sampled_labels])
    
    # Rescale images from [-1, 1] to [0, 1] for plotting
    gen_imgs = 0.5 * gen_imgs + 0.5
    
    fig, axs = plt.subplots(r, c, figsize=(10,4))
    fig.suptitle(f"Images at epoch {epoch}", fontsize=16)
    cnt = 0
    for i in range(r):
        for j in range(c):
            axs[i,j].imshow(gen_imgs[cnt])
            axs[i,j].set_title(f"Class: {sampled_labels[cnt][0]}")
            axs[i,j].axis('off')
            cnt += 1
    plt.show()
    plt.close()

def train_cgan(epochs, batch_size=128, sample_interval=1000):
    #TensorBoard callback for logging
    log_dir = "logs/adversarial/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
    summary_writer = tf.summary.create_file_writer(log_dir)
    # Ground truth labels for real (1) and fake (0) images
    valid = np.ones((batch_size, 1))
    fake = np.zeros((batch_size, 1))

    for epoch in range(epochs):

        # ---------------------
        #  Train Discriminator
        # ---------------------
        # Select a random batch of real images and their labels
        idx = np.random.randint(0, x_train.shape[0], batch_size)
        real_imgs, labels = x_train[idx], y_train[idx]

        # Generate a batch of fake images with the same labels
        noise = np.random.normal(0, 1, (batch_size, LATENT_DIM))
        gen_imgs = generator.predict([noise, labels])

        # Train the discriminator on separate batches of real and fake images
        d_loss_real = discriminator.train_on_batch([real_imgs, labels], valid)
        d_loss_fake = discriminator.train_on_batch([gen_imgs, labels], fake)
        d_loss = 0.5 * np.add(d_loss_real, d_loss_fake)

        # -----------------
        #  Train Generator
        # -----------------
        # Generate new noise and a batch of RANDOM labels to train the generator
        noise = np.random.normal(0, 1, (batch_size, LATENT_DIM))
        sampled_labels = np.random.randint(0, NUM_CLASSES, batch_size).reshape(-1, 1)

        # Train the generator (via the combined cgan model) to make the
        # discriminator think the fake images are real (output a '1').
        g_loss = cgan.train_on_batch([noise, sampled_labels], valid)

        # Print progress and show sample images at intervals
        if epoch % 100 == 0:
            print(f"{epoch} [D loss: {d_loss[0]:.4f}, acc.: {100*d_loss[1]:.2f}%] [G loss: {g_loss:.4f}]")
        
        if epoch % sample_interval == 0:
            sample_images(epoch, generator, LATENT_DIM)

# --- 7. Start Training ---
# NOTE: Adversarial GANs can be unstable and may require many epochs to
# produce good results. Monitor the sampled images. If they remain noise
# after ~20,000 epochs, the model has likely failed to converge.
#train_cgan(epochs=50000, batch_size=64, sample_interval=2000)

#### Visualization of the Training Data using TensorBoard


In [10]:
# Load the TensorBoard notebook extension
# %load_ext tensorboard

# Launch TensorBoard, pointing it to the directory you just downloaded
%tensorboard --logdir /home/taz/Documents/Masters/DeepLearning/Deep-Learning/DataAugmentationConditionalGan/kaggle_outputs/logs/

Reusing TensorBoard on port 6009 (pid 66765), started 0:00:10 ago. (Use '!kill 66765' to kill it.)